In [0]:
import requests

In [0]:
# Base URL for the dataset
base_url = "https://s3.amazonaws.com/nyc-tlc/trip+data"

# Example: Construct URL for January 2023
year = "2023"
month = "01"
file_type = "csv"
dataset_url = f"{base_url}/yellow_tripdata_{year}-{month}.{file_type}"

print(f"Dataset URL: {dataset_url}")


In [0]:
# Define the destination path in Databricks FileStore
destination_path = f"/dbfs/tmp/nyc_tlc/yellow_tripdata_{year}-{month}.{file_type}"

# Use Databricks Utilities to download the file
dbutils.fs.mkdirs("/tmp/nyc_tlc")  # Create a directory if it doesn't exist
dbutils.fs.cp(dataset_url, destination_path)

print(f"File downloaded to: {destination_path}")


In [0]:
from pyspark.sql import SparkSession

# Load the data into a Spark DataFrame
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(destination_path)

# Show a sample of the data
df.show()


In [0]:
# Specify the years and months
years = ["2023", "2022"]
months = [f"{i:02d}" for i in range(1, 13)]  # Generates ["01", "02", ..., "12"]

# Base URL and file type
base_url = "https://s3.amazonaws.com/nyc-tlc/trip+data"
file_type = "csv"
destination_dir = "/dbfs/tmp/nyc_tlc"

dbutils.fs.mkdirs(destination_dir)

# Loop through years and months to download files
for year in years:
    for month in months:
        dataset_url = f"{base_url}/yellow_tripdata_{year}-{month}.{file_type}"
        destination_path = f"{destination_dir}/yellow_tripdata_{year}-{month}.{file_type}"
        try:
            dbutils.fs.cp(dataset_url, destination_path)
            print(f"Downloaded: {dataset_url}")
        except Exception as e:
            print(f"Failed to download {dataset_url}: {e}")

# Load all downloaded files into a single DataFrame
all_data_path = f"{destination_dir}/*.csv"
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(all_data_path)

# Show data
df.show()


In [0]:
# Save as Delta Lake for future use
delta_path = "/mnt/delta/nyc_tlc_data"
df.write.format("delta").mode("overwrite").save(delta_path)

print(f"Data saved to Delta Lake: {delta_path}")


In [0]:
import requests
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import time

# Initialize Spark Session
spark = SparkSession.builder.appName("Free API Data Extraction").getOrCreate()

# Base configuration for API
base_url = "https://api.example.com/data"  # Replace with your API URL
auth_url = "https://api.example.com/auth"  # Replace with the token endpoint
client_id = "your_client_id"               # Replace with your client ID
client_secret = "your_client_secret"       # Replace with your client secret

# Function to get the Bearer token
def get_bearer_token():
    payload = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret
    }
    response = requests.post(auth_url, data=payload)
    response.raise_for_status()  # Raise exception if the request fails
    token_data = response.json()
    return token_data.get("access_token")

# Function to fetch paginated data
def fetch_paginated_data(api_url, headers, params=None):
    all_data = []
    page = 1
    while True:
        if params is None:
            params = {}
        params['page'] = page  # Add pagination parameter
        response = requests.get(api_url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        all_data.extend(data['results'])  # Adjust key based on API response structure
        if not data.get('next'):  # If no 'next' key, stop
            break
        page += 1
        time.sleep(1)  # Respect rate limits
    return all_data

# Function to process data for multiple countries/years
def fetch_data_for_all_conditions(countries, years):
    bearer_token = get_bearer_token()  # Fetch the token
    headers = {"Authorization": f"Bearer {bearer_token}"}
    combined_data = []

    for country in countries:
        for year in years:
            print(f"Fetching data for Country: {country}, Year: {year}")
            params = {"country": country, "year": year}
            data = fetch_paginated_data(base_url, headers, params=params)
            combined_data.extend(data)

    return combined_data

# Configuration for countries and years
countries = ["USA", "UK", "India"]  # Replace with the list of countries
years = ["2021", "2022", "2023"]   # Replace with the range of years

# Fetch the data
all_data = fetch_data_for_all_conditions(countries, years)

# Convert data to Spark DataFrame
rdd = spark.sparkContext.parallelize(all_data)
df = spark.read.json(rdd)

# Show the DataFrame
df.show()

# Save the DataFrame to Delta Lake for further processing
output_path = "/mnt/delta/free_api_data"
df.write.format("delta").mode("overwrite").save(output_path)

print(f"Data saved to Delta Lake at: {output_path}")


In [0]:
rdd = spark.sparkContext.parallelize(all_data)
df = spark.read.json(rdd)


In [0]:
df.write.format("delta").mode("overwrite").save(output_path)


In [0]:
try:
    response = requests.get(api_url, headers=headers, params=params)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"Error fetching data: {e}")


**NYC taxi data**

In [0]:
import requests
from pyspark.sql import SparkSession
import json

def fetch_data(api_url, params=None):
    response = requests.get(api_url, params=params)
    if response.status_code == 200:
        return response.json()  
    else:
        raise Exception(f"API Request Failed: {response.status_code}, {response.text}")

api_url = "https://data.cityofnewyork.us/resource/2yzn-sicd.json"

params = {
    "$limit": 1000 
}

data = fetch_data(api_url, params)

In [0]:
rdd = spark.sparkContext.parallelize(data)
df = spark.read.json(rdd)

print("Sample data from the API:")
df.show(truncate=False)

output_path = "/mnt/delta/nyc_taxi_data"
df.write.format("delta").mode("overwrite").save(output_path)

print(f"Data saved to Delta Lake at: {output_path}")


In [0]:
def fetch_all_data(api_url, rows_per_page=1000):
    all_data = []
    offset = 0

    while True:
        params = {"$limit": rows_per_page, "$offset": offset}
        data = fetch_data(api_url, params)
        
        if not data:  
            break
        
        all_data.extend(data)
        offset += rows_per_page

    return all_data

full_data = fetch_all_data(api_url)

rdd_full = spark.sparkContext.parallelize(full_data)
df_full = spark.read.json(rdd_full)
df_full.show(truncate=False)

df_full.write.format("delta").mode("overwrite").save(output_path)


In [0]:
import requests

token_url = "https://api.example.com/auth/token"
client_id = "your_client_id"
client_secret = "your_client_secret"

data = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
}

response = requests.post(token_url, data=data)

if response.status_code == 200:
    token = response.json().get("access_token")
    print(f"Bearer Token: {token}")
else:
    print(f"Failed to get token: {response.status_code}, {response.text}")


In [0]:
headers = {
    "Authorization": f"Bearer {token}",
}

api_url = "https://api.example.com/data"
response = requests.get(api_url, headers=headers)
print(response.json())


In [0]:
import requests
import pandas as pd
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("NYC_Taxi_Data_Extraction").getOrCreate()

# API endpoint
api_url = "https://data.cityofnewyork.us/resource/2yzn-sicd.json"

# Define years and months to loop through
years = range(2020, 2023)  # Adjust as needed
months = range(1, 13)

# Function to fetch data for a given year and month
def fetch_data(year, month):
    # Format the month as MM
    month_str = f"{month:02d}"
    # Construct $where filter for year and month
    start_date = f"{year}-{month_str}-01T00:00:00"
    if month == 12:
        end_date = f"{year + 1}-01-01T00:00:00"
    else:
        end_date = f"{year}-{month + 1:02d}-01T00:00:00"
    
    params = {
        "$where": f"pickup_datetime >= '{start_date}' AND pickup_datetime < '{end_date}'",
        "$limit": 1000  # Adjust based on API limits or pagination needs
    }
    
    response = requests.get(api_url, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Failed to fetch data for {year}-{month_str}: {response.status_code}")
        return []

# Loop through years and months, fetch data, and combine into a list
all_data = []

for year in years:
    for month in months:
        print(f"Fetching data for {year}-{month:02d}...")
        monthly_data = fetch_data(year, month)
        all_data.extend(monthly_data)

# Convert data to Pandas DataFrame for processing
df = pd.DataFrame(all_data)

# Convert Pandas DataFrame to PySpark DataFrame
spark_df = spark.createDataFrame(df)

# Show sample data
spark_df.show()

# Save the data to Delta Lake or any other format
output_path = "/mnt/delta/nyc_taxi_data"
spark_df.write.format("delta").mode("overwrite").save(output_path)

print(f"Data saved to Delta Lake at: {output_path}")


In [0]:
import requests
from datetime import datetime
import pandas as pd

# API Endpoint
api_url = "https://data.cityofnewyork.us/resource/2yzn-sicd.json"

# Function to fetch data for a specific month and year
def fetch_monthly_data(api_url, year, month, limit=1000, offset=0):
    # Format dates for filtering
    start_date = f"{year}-{month:02d}-01T00:00:00"
    if month == 12:
        end_date = f"{year + 1}-01-01T00:00:00"
    else:
        end_date = f"{year}-{month + 1:02d}-01T00:00:00"
    
    # Query parameters
    params = {
        "$where": f"pickup_datetime >= '{start_date}' AND pickup_datetime < '{end_date}'",
        "$limit": limit,
        "$offset": offset
    }
    
    # API Request
    response = requests.get(api_url, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Failed to fetch data: {response.status_code}, {response.text}")
        return []

# Function to fetch data for a specific year
def fetch_yearly_data(api_url, year, limit=1000):
    all_data = []
    for month in range(1, 13):  # Loop through all months
        print(f"Fetching data for {year}-{month:02d}...")
        offset = 0
        while True:
            # Fetch data for the month
            monthly_data = fetch_monthly_data(api_url, year, month, limit, offset)
            if not monthly_data:
                break
            all_data.extend(monthly_data)
            offset += limit  # Increment offset for pagination
    return all_data

# Loop through multiple years and fetch data
def fetch_data_for_years(api_url, start_year, end_year, limit=1000):
    all_data = []
    for year in range(start_year, end_year + 1):  # Loop through years
        yearly_data = fetch_yearly_data(api_url, year, limit)
        all_data.extend(yearly_data)
    return all_data

# Fetch data for 2020 and 2021 as an example
data = fetch_data_for_years(api_url, 2020, 2021)

# Convert to Pandas DataFrame for analysis
df = pd.DataFrame(data)
print(f"Fetched {len(df)} records.")
print(df.head())

# Save to CSV
output_path = "nyc_taxi_data_2020_2021.csv"
df.to_csv(output_path, index=False)
print(f"Data saved to {output_path}")


In [0]:
def fetch_all_paginated_data(year, month):
    all_month_data = []
    offset = 0
    limit = 1000
    while True:
        params = {
            "$where": f"pickup_datetime >= '{year}-{month:02d}-01T00:00:00' AND "
                      f"pickup_datetime < '{year}-{month + 1:02d}-01T00:00:00'",
            "$limit": limit,
            "$offset": offset
        }
        response = requests.get(api_url, params=params)
        if response.status_code == 200:
            data = response.json()
            if not data:
                break
            all_month_data.extend(data)
            offset += limit
        else:
            print(f"Failed to fetch data: {response.status_code}, {response.text}")
            break
    return all_month_data
